# 排序网络集合

通常，我们会记录一组 `Networks` 数据，同时改变某个其他变量，例如电压、电流或时间。因此……现在你已经有了这组数据，并且想要查看某个特征如何演变，或者计算一些具有代表性的统计数据。这个示例演示了如何使用 [NetworkSets](../../tutorials/NetworkSet.ipynb) 来实现这一点。

## 生成一些数据

为了本示例的目的，我们使用预定义的 `skrf.media.Media` 对象来生成一些网络，并将它们保存为一系列 Touchstone 文件。每个文件都以时间戳命名，时间戳由便捷函数 `rf.util.now_string()` 生成。

In [ ]:
from time import sleep

import matplotlib.pyplot as plt
import numpy as np

import skrf as rf
from skrf.util import now_string, now_string_2_dt

%matplotlib inline
rf.stylely()


!rm -rf tmp
!mkdir tmp

wg = rf.instances.wr10 # just a dummy media object to generate data

for k in range(10):
    # timestamp generated with `now_string()`
    ntwk = wg.random(name=now_string()+'.s1p')
    ntwk.s = k*ntwk.s
    ntwk.write_touchstone(dir='tmp')
    sleep(.1)

让我们看看我们都做了些什么。

In [ ]:
!ls tmp

## 未排序（默认）

当使用 `NetworkSet.from_dir()` 创建时，`NetworkSet` 会随机存储每个 `Network`。

In [ ]:
ns = rf.networkSet.NetworkSet.from_dir('tmp')
ns.ntwk_set

## 排序

In [ ]:
ns.sort()
ns.ntwk_set

## 使用 `key` 参数进行排序

您还可以通过 `key` 参数传递一个函数，从而可以根据任意属性进行排序。例如，我们可以根据名称的秒数部分进行排序。

In [ ]:
ns = rf.networkSet.NetworkSet.from_dir('tmp')
ns.sort(key = lambda x: x.name.split('.')[0])
ns.ntwk_set

## 提取日期和时间

您还可以将 ntwk 名称转换为 datetime 对象，以便您可以使用 pandas 绘制图形或进行其他处理。`rf.util.now_string()` 函数有一个配套函数，即 `rf.util.now_string_2_dt()`。多么有创意啊……

In [ ]:
ns.sort()
dt_idx = [now_string_2_dt(k.name ) for k in ns]
dt_idx

## 放入 Pandas DataFrame 并进行绘图

下一步是将网络集沿时间轴进行切片。例如，我们可能希望查看几个不同频率下的 S11 相位。可以使用以下脚本来实现。请注意，NetworkSets 可以像 Networks 一样，使用人类可读的字符串按频率进行切片。

In [ ]:
import pandas as pd

dates = pd.DatetimeIndex(dt_idx)

# create a function to pull out S11 in degrees at a specific frequency

s_deg_at = lambda s:{s: [k[s].s_deg[0,0,0] for k in ns]}  # noqa: E731

for f in ['80ghz', '90ghz','100ghz']:
    df =pd.DataFrame(s_deg_at(f), index=dates)
    df.plot(ax=plt.gca())
plt.title('Phase Evolution in Time')
plt.ylabel('S11 (deg)')

## 使用 `signature` 可视化行为

将网络参数集在所有频率上的标量分量的演变过程可视化可能是有用的。这可以通过一些数组操作和 `imshow` 函数来实现。例如，如果我们对每个网络取其以 dB 为单位的幅度，并由此创建一个二维矩阵，

In [ ]:
mat = np.array([k.s_db.flatten() for k in ns])
mat.shape

这个数组的形状为（“网络数量”，“频率点数量”）。可以使用 imshow 进行可视化。下面的大部分代码只是添加标签和调整坐标轴刻度。

In [ ]:
freq = ns[0].frequency

# creates x and y scales
extent = [freq.f_scaled[0], freq.f_scaled[-1], len(ns) ,0]

#make the image
plt.imshow(mat, aspect='auto',extent=extent,interpolation='nearest')

# label things
plt.grid(0)
freq.labelXAxis()
plt.ylabel('Network #')
cbar = plt.colorbar()
cbar.set_label('Magnitude (dB)')

此过程通过 `NetworkSet.signature()` 方法实现自动化。它甚至有一个 `vs_time` 参数，如果网络名称是由 `rf.now_string()` 编写的，该参数将自动创建 DateTime 索引。

In [ ]:
ns.signature(component='s_db', vs_time=True,cbar_label='Magnitude (dB)')